# IBM Applied Data Science Capstone: SpaceX Falcon 9 First Stage Landing Prediction
## Lab 7: Machine Learning Prediction & Classification
**Author:** Shravani D  
**GitHub Repository:** [SpaceX-Falcon9-Landing-Prediction](https://github.com/ShravaniD/SpaceX-Falcon9-Landing-Prediction)

---
### Objective
Build, train, tune, and evaluate four machine learning classification algorithms:
1. **Logistic Regression**
2. **Support Vector Machine (SVM)**
3. **Decision Tree Classifier**
4. **K-Nearest Neighbors (KNN)**

Using `GridSearchCV` with 10-fold cross-validation to predict first-stage booster landing outcomes (`Class` = 1 or 0).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report
import os

sns.set_theme(style="white")
df2 = pd.read_csv(os.path.join("..", "data", "dataset_part_2.csv"))
X = pd.read_csv(os.path.join("..", "data", "dataset_part_3.csv"))
Y = df2['Class'].to_numpy()

print(f"Features matrix X shape: {X.shape}")
print(f"Target vector Y shape: {Y.shape}")


### Step 1: Feature Standardization & Train/Test Split


In [1]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=2)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")


### Model 1: Logistic Regression


In [1]:
lr = LogisticRegression(random_state=42)
lr_params = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr_cv = GridSearchCV(lr, lr_params, cv=10)
lr_cv.fit(X_train, Y_train)

print("Tuned Hyperparameters:", lr_cv.best_params_)
print(f"Train Accuracy: {lr_cv.best_score_*100:.2f}%")
print(f"Test Accuracy: {lr_cv.score(X_test, Y_test)*100:.2f}%")


### Model 2: Support Vector Machine (SVM)


In [1]:
svm = SVC(random_state=42)
svm_params = {'kernel': ['linear', 'rbf', 'sigmoid'], 'C': np.logspace(-2, 2, 5)}
svm_cv = GridSearchCV(svm, svm_params, cv=10)
svm_cv.fit(X_train, Y_train)

print("Tuned Hyperparameters:", svm_cv.best_params_)
print(f"Train Accuracy: {svm_cv.best_score_*100:.2f}%")
print(f"Test Accuracy: {svm_cv.score(X_test, Y_test)*100:.2f}%")


### Model 3: Decision Tree Classifier


In [1]:
tree = DecisionTreeClassifier(random_state=42)
tree_params = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [2, 4, 6, 8, 10],
    'max_features': ['sqrt', 'log2'],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10]
}
tree_cv = GridSearchCV(tree, tree_params, cv=10)
tree_cv.fit(X_train, Y_train)

print("Tuned Hyperparameters:", tree_cv.best_params_)
print(f"Train Accuracy: {tree_cv.best_score_*100:.2f}%")
print(f"Test Accuracy: {tree_cv.score(X_test, Y_test)*100:.2f}%")


### Model 4: K-Nearest Neighbors (KNN)


In [1]:
knn = KNeighborsClassifier()
knn_params = {'n_neighbors': list(range(1, 11)), 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 'p': [1, 2]}
knn_cv = GridSearchCV(knn, knn_params, cv=10)
knn_cv.fit(X_train, Y_train)

print("Tuned Hyperparameters:", knn_cv.best_params_)
print(f"Train Accuracy: {knn_cv.best_score_*100:.2f}%")
print(f"Test Accuracy: {knn_cv.score(X_test, Y_test)*100:.2f}%")


### Model Evaluation & Comparison


In [1]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Support Vector Machine', 'Decision Tree', 'K-Nearest Neighbors'],
    'Train Accuracy (10-Fold CV)': [lr_cv.best_score_, svm_cv.best_score_, tree_cv.best_score_, knn_cv.best_score_],
    'Test Accuracy': [lr_cv.score(X_test, Y_test), svm_cv.score(X_test, Y_test), tree_cv.score(X_test, Y_test), knn_cv.score(X_test, Y_test)]
})
results['Train Accuracy (10-Fold CV)'] = (results['Train Accuracy (10-Fold CV)'] * 100).round(2).astype(str) + '%'
results['Test Accuracy'] = (results['Test Accuracy'] * 100).round(2).astype(str) + '%'
results


### Final Predictive Insights
- All four models achieved an identical test accuracy of **83.33%** on the hold-out test set (15/18 correct predictions).
- **Decision Tree Classifier** achieved the highest cross-validation score on the training data (87.50%), demonstrating exceptional capability to capture non-linear feature interactions (such as FlightNumber thresholds and booster reuse counts).
- **Logistic Regression & SVM** demonstrated identical confusion matrices (3 True Negatives, 3 False Positives, 0 False Negatives, 12 True Positives), with a 100% recall for successful landings.
